# TMDb Keyword Download

Downloads the TMDb daily keyword export — no API key required.

| | |
|---|---|
| **Source** | `https://files.tmdb.org/p/exports/keyword_ids_MM_DD_YYYY.json.gz` |
| **Format** | Newline-delimited JSON `{"id": int, "name": str}` |
| **Output** | `data/tmdb_keywords_canonical.csv` — columns: `tmdb_keyword_id`, `name` |

Retries the last 3 days in case today's export hasn't been published yet.

## TMDb Keyword Guidelines

> Source: [TMDb Bible — Keywords](https://www.themoviedb.org/bible/movie/59f3b16d9251414f20000007)

Keywords describe **plot themes** — choose only the few best (no spoilers).  
Around **5–10 for TV shows** and **15–20 maximum for movies** is reasonable.

### Not a keyword
- Quotes, taglines, titles, actors, characters, crew, writers, networks, companies, award ceremonies
  - Narrow exceptions: recurring mythical figures (e.g. `santa claus`), biopic subjects, titles that are thematically relevant (e.g. `jungle`), documentaries *about* an award ceremony
- Genres re-added as keywords (e.g. `comedy` on a comedy) — **exception:** `family` keyword ≠ Family genre; use it when family is a strong theme
- Trivia keywords (e.g. `face slap`, `one word title`) — use lists instead
  - Pre-approved trivia: `3d`, `based on book or novel`, `woman director`, `duringcreditsstinger`
- The release year/decade itself (already in release date)

### Rules
- **Objective only** — no opinion or judgment
- **English only** — translated keywords not yet supported
- **Reuse existing** — avoid duplicates for plurals/alternate spellings (`teenager` not `teen` or `teens`)
- Year/decade/century keywords are allowed when the story is set in the past or future *relative to release year*
- Pornographic keywords only on entries with `adult: true`
- Do not copy IMDb keywords wholesale; do not delete valid keywords

In [1]:
import gzip
import io
import json
import urllib.error
import urllib.request
from dataclasses import dataclass, field
from datetime import date, timedelta
from pathlib import Path
from typing import Iterator

import pandas as pd


@dataclass(frozen=True)
class Config:
    base_url: str = "https://files.tmdb.org/p/exports"
    export_name: str = "keyword_ids"
    lookback_days: int = 3
    timeout_seconds: int = 60
    output_file: Path = field(default_factory=lambda: Path("data/tmdb_keywords_canonical.csv"))


def candidate_urls(cfg: Config) -> Iterator[tuple[date, str]]:
    """Yield (date, url) for today through lookback_days ago."""
    for delta in range(cfg.lookback_days):
        d = date.today() - timedelta(days=delta)
        url = f"{cfg.base_url}/{cfg.export_name}_{d.month:02d}_{d.day:02d}_{d.year}.json.gz"
        yield d, url


def fetch_export(cfg: Config) -> tuple[date, bytes]:
    """Download the most recent available export; raise RuntimeError if all fail."""
    errors: list[str] = []
    for d, url in candidate_urls(cfg):
        print(f"Trying {url} ...", end=" ", flush=True)
        try:
            with urllib.request.urlopen(url, timeout=cfg.timeout_seconds) as resp:
                data = resp.read()
            print(f"{len(data):,} bytes")
            return d, data
        except urllib.error.HTTPError as exc:
            msg = f"HTTP {exc.code}"
            print(msg)
            errors.append(f"{d}: {msg}")
        except urllib.error.URLError as exc:
            msg = str(exc.reason)
            print(msg)
            errors.append(f"{d}: {msg}")
    raise RuntimeError(
        f"Failed to fetch keyword export for last {cfg.lookback_days} days:\n"
        + "\n".join(errors)
    )


def parse_export(raw_gz: bytes) -> pd.DataFrame:
    """Decompress and parse newline-delimited JSON into a DataFrame."""
    with gzip.open(io.BytesIO(raw_gz), "rt", encoding="utf-8") as fh:
        records = [
            {"tmdb_keyword_id": obj["id"], "name": obj["name"]}
            for line in fh
            if (s := line.strip())
            for obj in [json.loads(s)]
        ]
    return (
        pd.DataFrame(records)
        .drop_duplicates("tmdb_keyword_id")
        .sort_values("tmdb_keyword_id")
        .reset_index(drop=True)
        .astype({"tmdb_keyword_id": "int64", "name": "string"})
    )


print("Functions defined — run next cell to download")

Functions defined — run next cell to download


In [2]:
cfg = Config()
cfg.output_file.parent.mkdir(parents=True, exist_ok=True)

export_date, raw_gz = fetch_export(cfg)
kw_canonical = parse_export(raw_gz)

kw_canonical.to_csv(cfg.output_file, index=False)
print(f"\nExport date : {export_date}")
print(f"Keywords    : {len(kw_canonical):,}")
print(f"Saved to    : {cfg.output_file}")
kw_canonical.head(10)

Trying https://files.tmdb.org/p/exports/keyword_ids_03_11_2026.json.gz ... 

910,465 bytes



Export date : 2026-03-11
Keywords    : 84,842
Saved to    : data/tmdb_keywords_canonical.csv


,tmdb_keyword_id,name
0,30,individual
1,65,holiday
2,74,germany
3,75,gunslinger
4,83,saving the world
5,90,"paris, france"
6,100,slum
7,107,"barcelona, spain"
8,108,transvestism
9,110,"venice, italy"


## Join to keyword lexicon

Adds `tmdb_keyword_id` to `output/tmdb_keyword_lexicon.csv` by exact-match on lowercase name.

> **Skipped automatically** when the lexicon file doesn't exist (e.g. first run or Kaggle).

In [3]:
LEXICON_FILE = Path("output/tmdb_keyword_lexicon.csv")

if not LEXICON_FILE.exists():
    print(f"Skipping join: {LEXICON_FILE} not found")
else:
    lexicon = pd.read_csv(LEXICON_FILE)

    if "tmdb_keyword_id" in lexicon.columns:
        lexicon = lexicon.drop(columns=["tmdb_keyword_id"])

    canonical = kw_canonical.copy()
    canonical["name_norm"] = canonical["name"].str.lower().str.strip()
    lexicon["name_norm"] = lexicon["keyword"].str.lower().str.strip()

    lexicon = (
        lexicon
        .merge(canonical[["tmdb_keyword_id", "name_norm"]], on="name_norm", how="left")
        .drop(columns="name_norm")
    )

    other_cols = [c for c in lexicon.columns if c not in ("keyword", "tmdb_keyword_id")]
    lexicon = lexicon[["keyword", "tmdb_keyword_id", *other_cols]]

    total = lexicon["keyword"].notna().sum()
    matched = lexicon["tmdb_keyword_id"].notna().sum()
    print(f"Matched   : {matched:,} / {total:,} ({matched / total * 100:.1f}%)")
    print(f"Unmatched : {total - matched:,}")

    lexicon.to_csv(LEXICON_FILE, index=False)
    print(f"Updated   : {LEXICON_FILE}")
    lexicon.head(5)

Matched   : 110,610 / 117,064 (94.5%)
Unmatched : 6,454


Updated   : output/tmdb_keyword_lexicon.csv


## EDA — Keyword Overview

In [4]:
import plotly.express as px

df = kw_canonical.copy()
df["name_len"] = df["name"].str.len()
df["word_count"] = df["name"].str.split().str.len()

print(f"Total keywords : {len(df):,}")
print(f"ID range       : {df['tmdb_keyword_id'].min():,} – {df['tmdb_keyword_id'].max():,}")
print(f"\nName length (chars):")
print(df["name_len"].describe().round(1).to_string())
print(f"\nWord count:")
print(df["word_count"].describe().round(1).to_string())

Total keywords : 84,842
ID range       : 30 – 369,532

Name length (chars):
count    84842.0
mean        11.8
std          6.0
min          1.0
25%          8.0
50%         11.0
75%         15.0
max        150.0

Word count:
count    84842.0
mean         1.8
std          0.9
min          1.0
25%          1.0
50%          2.0
75%          2.0
max         25.0


In [5]:
# ID space: assigned vs gaps
full_range = int(df["tmdb_keyword_id"].max() - df["tmdb_keyword_id"].min() + 1)
assigned = len(df)

fig = px.bar(
    x=["Assigned", "Gaps (deleted / never used)"],
    y=[assigned, full_range - assigned],
    color=["Assigned", "Gaps"],
    color_discrete_map={"Assigned": "#1f77b4", "Gaps": "#d62728"},
    text_auto=True,
    title=f"TMDb keyword ID space  (range 1–{full_range:,})",
    labels={"x": "", "y": "Count"},
)
fig.update_layout(showlegend=False)
fig.show()

In [6]:
# Name length distribution
fig = px.histogram(
    df,
    x="name_len",
    nbins=60,
    title="Keyword name length distribution (characters)",
    labels={"name_len": "Characters", "count": "Keywords"},
)
fig.update_layout(bargap=0.05)
fig.show()

In [7]:
# Word count breakdown
wc = (
    df["word_count"].value_counts()
    .sort_index()
    .reset_index()
    .rename(columns={"word_count": "words", "count": "keywords"})
)
wc["pct"] = (wc["keywords"] / wc["keywords"].sum() * 100).round(1)

fig = px.bar(
    wc[wc["words"] <= 8],
    x="words",
    y="keywords",
    text="pct",
    title="Keywords by word count",
    labels={"words": "Word count", "keywords": "Keywords"},
)
fig.update_traces(texttemplate="%{text}%", textposition="outside")
fig.update_layout(xaxis=dict(tickmode="linear"))
fig.show()

In [8]:
# Keyword density by ID bucket (proxy for when keywords were added)
df["id_bucket"] = (df["tmdb_keyword_id"] // 10_000) * 10_000
bucket_counts = df.groupby("id_bucket").size().reset_index(name="count")

fig = px.bar(
    bucket_counts,
    x="id_bucket",
    y="count",
    title="Keyword density by ID range (10k buckets)",
    labels={"id_bucket": "ID range start", "count": "Keywords assigned"},
)
fig.show()

In [9]:
# Top 30 first words — genre / theme signals
first_words = (
    df["name"].str.split().str[0].str.lower()
    .value_counts()
    .head(30)
    .reset_index()
    .rename(columns={"name": "word", "count": "count"})
)

fig = px.bar(
    first_words,
    x="count",
    y="word",
    orientation="h",
    title="Top 30 first words in keyword names",
    labels={"word": "", "count": "Keywords"},
)
fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()

## Keyword Taxonomy

Rule-based classification of the 84k keyword corpus into semantic categories, plus a surface of **rule-breakers** — keywords that stretch or violate the TMDb guidelines.

| Category | Signal for emotion scoring |
|---|---|
| `plot_event` | very high |
| `character_role` | high |
| `theme` | very high |
| `trope` | high |
| `location` | medium |
| `time_period` | low |
| `object_motif` | medium |
| `mythic_figure` | medium |
| `production_meta` | none |
| `genre_as_keyword` ⚠️ | violates guidelines |

In [10]:
import re
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd

df = kw_canonical.copy()

# ── Rule sets ──────────────────────────────────────────────────────────────

TMDB_GENRES = {
    "action", "adventure", "animation", "comedy", "crime", "documentary",
    "drama", "family", "fantasy", "history", "horror", "music", "mystery",
    "romance", "science fiction", "thriller", "tv movie", "war", "western",
}

# Subgenres / genre labels not in the TMDb official list
GENRE_SUBGENRES = {
    "musical", "dark comedy", "romantic comedy", "stand-up comedy",
    "softcore", "gay theme", "martial arts", "slasher", "grindhouse",
    "erotic thriller", "sexploitation", "blaxploitation", "giallo",
    "spaghetti western", "neo-noir", "mockbuster", "pink film",
    "social drama", "road movie", "heist film", "courtroom drama",
    "coming-of-age film", "buddy film", "screwball comedy",
    "film noir", "psychological thriller", "parody", "satire",
    "boys' love (bl)", "music documentary", "nature documentary",
    "sports documentary", "concert documentary", "erotic movie",
    "political thriller", "supernatural thriller", "legal thriller",
}

PRODUCTION_META = {
    # Source material
    "3d", "duringcreditsstinger", "woman director",
    "based on book or novel", "based on novel", "based on comic",
    "based on true story", "based on short story", "based on play",
    "based on play or musical", "based on video game", "based on manga",
    "based on webtoon", "based on web novel", "based on light novel",
    # Film form / format / technique
    "short film", "short", "silent film", "found footage", "black and white",
    "stop motion", "anime", "animated film", "experimental",
    "mockumentary", "claymation", "rotoscoping", "hand drawn",
    "concert film", "concert", "biography", "biopic",
    "documentary film", "educational", "educational film",
    "cartoon", "live performance", "avant-garde",
    "interview", "student film", "independent film",
    "lost film", "pre-code", "behind the scenes", "making of",
    "theater play", "stage production",
    # Franchise / release meta
    "sequel", "prequel", "remake", "reboot", "spin-off", "anthology",
    "crossover", "direct-to-video",
}

MYTHIC_FIGURES = {
    "santa claus", "dracula", "frankenstein", "king arthur", "robin hood",
    "hercules", "sherlock holmes", "zeus", "medusa", "odysseus",
    "cinderella", "snow white", "sleeping beauty", "pinocchio",
}

TIME_REGEXES = [
    r"\b\d{4}s\b",                          # 1980s
    r"\b\d{1,2}(?:st|nd|rd|th) century\b",  # 19th century
    r"\bworld war\b",
    r"\b(?:ancient|medieval|victorian|renaissance|prehistoric)\b",
    r"\b(?:future|post-apocalyptic|dystopian|distant future)\b",
    r"\bcold war\b",
]

# Countries, regions, continents, and cities as standalone keywords
COUNTRIES = {
    "africa", "asia", "europe", "latin america", "middle east",
    "france", "england", "germany", "italy", "japan", "russia", "spain",
    "mexico", "india", "china", "australia", "canada", "brazil", "korea",
    "sweden", "denmark", "norway", "ireland", "scotland", "wales", "poland",
    "portugal", "netherlands", "belgium", "argentina", "colombia", "turkey",
    "thailand", "indonesia", "philippines", "nigeria", "egypt", "iran",
    "pakistan", "bangladesh", "vietnam", "ukraine", "greece", "romania",
    "california", "texas", "new york", "florida", "hawaii",
    "hong kong", "taiwan",
}

LOCATION_REGEXES = [
    r", (?:france|england|germany|italy|japan|russia|spain|mexico|india|china|australia|usa|uk|canada)",
    r"\b(?:los angeles|new york city|new york|paris|london|tokyo|berlin|rome|moscow|chicago|miami|hong kong)\b",
    r"\b(?:high school|prison|hospital|space station|haunted house|island|desert|jungle|ocean|mountain|castle|mansion|village|small town|big city)\b",
]

TROPE_PATTERNS = [
    r"\btime loop\b", r"\bidentity swap\b", r"\bsecret identity\b",
    r"\bdouble life\b", r"\bfish out of water\b", r"\bchosen one\b",
    r"\bcoming of age\b", r"\btrue love\b", r"\bredemption arc\b",
    r"\borigin story\b", r"\btwist ending\b", r"\bhero's journey\b",
]

CHARACTER_ROLE_WORDS = {
    "detective", "superhero", "villain", "assassin", "spy", "soldier",
    "teenager", "child", "orphan", "scientist", "doctor", "lawyer",
    "serial killer", "hitman", "hacker", "pilot", "reporter", "princess",
    "king", "queen", "knight", "wizard", "warrior", "outlaw", "prisoner",
    "single mother", "single father", "father", "mother",
}
# Pre-compile word-boundary patterns to avoid substring false positives
# e.g. "king" must not match "filmmaking", "mother" must not match "godmother"
_CHARACTER_ROLE_RE = re.compile(
    r"\b(?:" + "|".join(re.escape(w) for w in CHARACTER_ROLE_WORDS) + r")\b"
)

OBJECT_MOTIF_WORDS = {
    "spaceship", "robot", "sword", "gun", "mask", "map", "treasure",
    "artifact", "portal", "book", "letter", "photograph", "weapon",
    "machine", "device", "formula", "serum", "virus",
}
_OBJECT_MOTIF_RE = re.compile(
    r"\b(?:" + "|".join(re.escape(w) for w in OBJECT_MOTIF_WORDS) + r")\b"
)


# ── Classifier ─────────────────────────────────────────────────────────────

def classify(name: str) -> str:
    n = name.lower().strip()

    if n in TMDB_GENRES:
        return "genre_as_keyword"
    if n in GENRE_SUBGENRES:
        return "genre_as_keyword"
    if n in PRODUCTION_META or re.match(r"^based on", n):
        return "production_meta"
    if n in MYTHIC_FIGURES:
        return "mythic_figure"
    if n in COUNTRIES:
        return "location"
    if any(re.search(p, n) for p in TIME_REGEXES):
        return "time_period"
    if any(re.search(p, n) for p in LOCATION_REGEXES):
        return "location"
    if any(re.search(p, n) for p in TROPE_PATTERNS):
        return "trope"
    if _CHARACTER_ROLE_RE.search(n):
        return "character_role"
    if _OBJECT_MOTIF_RE.search(n):
        return "object_motif"
    # Themes: abstract single-word or short multi-word concepts
    word_count = len(n.split())
    if word_count <= 2:
        return "theme_or_plot"
    return "plot_event"


df["keyword_type"] = df["name"].apply(classify)

print(df["keyword_type"].value_counts().to_string())


keyword_type
theme_or_plot       70917
plot_event          11071
character_role       1028
location             1007
object_motif          378
time_period           262
production_meta        96
genre_as_keyword       53
trope                  18
mythic_figure          12


In [11]:
# ── Category distribution ───────────────────────────────────────────────────
cat_counts = df["keyword_type"].value_counts().reset_index()
cat_counts.columns = ["keyword_type", "count"]
cat_counts["pct"] = (cat_counts["count"] / cat_counts["count"].sum() * 100).round(1)

COLOR_MAP = {
    "theme_or_plot":    "#1f77b4",
    "plot_event":       "#aec7e8",
    "character_role":   "#ff7f0e",
    "location":         "#2ca02c",
    "time_period":      "#98df8a",
    "trope":            "#9467bd",
    "object_motif":     "#8c564b",
    "production_meta":  "#7f7f7f",
    "mythic_figure":    "#e377c2",
    "genre_as_keyword": "#d62728",
}

fig = px.bar(
    cat_counts,
    x="count",
    y="keyword_type",
    orientation="h",
    text="pct",
    color="keyword_type",
    color_discrete_map=COLOR_MAP,
    title="Keyword taxonomy distribution",
    labels={"keyword_type": "", "count": "Keywords"},
)
fig.update_traces(texttemplate="%{text}%", textposition="outside")
fig.update_layout(showlegend=False, yaxis={"categoryorder": "total ascending"})
fig.show()


In [12]:
# ── Rule-breakers & edge cases ─────────────────────────────────────────────

# 1. Genre words used as keywords
genre_kws = df[df["keyword_type"] == "genre_as_keyword"][["tmdb_keyword_id", "name"]]
print(f"Genre-as-keyword violations ({len(genre_kws)}):")
print(genre_kws.to_string(index=False))

# 2. Bare year keywords (rule: release year must not be a keyword)
year_kws = df[df["name"].str.match(r"^\d{4}$")]
print(f"\nBare year keywords ({len(year_kws)}) — likely rule violations:")
print(year_kws[["tmdb_keyword_id", "name"]].to_string(index=False))

# 3. Very short keywords (single/double char) — noise?
short_kws = df[df["name"].str.len() <= 2]
print(f"\nSuper-short keywords (≤2 chars, {len(short_kws)}):")
print(short_kws[["tmdb_keyword_id", "name"]].to_string(index=False))

# 4. Comma-containing keywords — possible multi-entry violations
# Note: location keywords like 'paris, france' are intentional
comma_kws = df[df["name"].str.contains(",", na=False) & (df["keyword_type"] != "location")]
print(f"\nNon-location comma keywords ({len(comma_kws)}) — possible multi-entries:")
print(comma_kws[["tmdb_keyword_id", "name"]].head(20).to_string(index=False))

# 5. All-caps — acronyms or noise?
allcaps = df[df["name"].str.match(r"^[A-Z]{2,}$")]
print(f"\nAll-caps keywords ({len(allcaps)}):")
print(allcaps[["tmdb_keyword_id", "name"]].to_string(index=False))

# 6. Keywords containing digits (not years/decades)
digit_kws = df[
    df["name"].str.contains(r"\d", na=False) &
    ~df["name"].str.match(r"^\d{4}$") &
    ~df["name"].str.match(r".*\d{4}s.*") &
    ~df["name"].str.match(r".*\d{1,2}(st|nd|rd|th) century.*")
]
print(f"\nKeywords with digits (not years/decades, {len(digit_kws)}):")
print(digit_kws[["tmdb_keyword_id", "name"]].head(30).to_string(index=False))


Genre-as-keyword violations (53):
 tmdb_keyword_id                   name
             779           martial arts
            4344                musical
            8201                 satire
            9716        stand-up comedy
            9755                 parody
            9807              film noir
            9840                romance
           10053          sexploitation
           10123            dark comedy
           11532             grindhouse
           12339                slasher
           12565 psychological thriller
           18035                 family
          155457       screwball comedy
          155477               softcore
          156212      spaghetti western
          159290     sports documentary
          159551              pink film
          167043             road movie
          190370           erotic movie
          207268               neo-noir
          207767        erotic thriller
          208879             mockbuster
      

In [13]:
# ── Treemap: full taxonomy at a glance ─────────────────────────────────────
fig = px.treemap(
    cat_counts,
    path=["keyword_type"],
    values="count",
    color="keyword_type",
    color_discrete_map=COLOR_MAP,
    title="Keyword taxonomy — treemap",
)
fig.update_traces(textinfo="label+value+percent root")
fig.show()


In [14]:
# ── Rule-breaker deep dive: genre violations ────────────────────────────────
# Show how many IDs per genre word — higher = used more 'wrongly'
rule_break_df = df[df["keyword_type"].isin(["genre_as_keyword"])].copy()
rule_break_df["label"] = rule_break_df["name"] + " (id=" + rule_break_df["tmdb_keyword_id"].astype(str) + ")"

# Also flag bare year keywords alongside genre violations
year_flag = df[df["name"].str.match(r"^\d{4}$")].copy()
year_flag["keyword_type"] = "bare_year"
year_flag["label"] = year_flag["name"]

combined = pd.concat([rule_break_df, year_flag], ignore_index=True)

fig = px.scatter(
    combined,
    x="tmdb_keyword_id",
    y="keyword_type",
    text="name",
    color="keyword_type",
    color_discrete_map={"genre_as_keyword": "#d62728", "bare_year": "#ff7f0e"},
    title="Rule-breakers by keyword ID",
    labels={"tmdb_keyword_id": "Keyword ID (proxy for age)", "keyword_type": ""},
    height=350,
)
fig.update_traces(textposition="top center", marker_size=10)
fig.update_layout(showlegend=False)
fig.show()


## Vocabulary Analysis — Common vs Cinema-Specific English

Using `wordfreq` Zipf frequency scores to classify each keyword's vocabulary:

| Zipf score | Meaning |
|---|---|
| ≥ 4 | very common (`the`, `time`, `love`) |
| 3–4 | common (`revenge`, `soldier`) |
| 2–3 | rare (`cyberpunk`, `heist`) |
| < 2 | very rare / unknown (`kaiju`, `xenomorph`) |

Keywords where **all tokens score < 3** are film-specific jargon, proper nouns, or invented terms — these are most likely to be **missing from sentiment lexicons**.

In [15]:
import re
from wordfreq import zipf_frequency

COMMON_THRESHOLD = 3.0  # Zipf score cutoff


def tokenize(name: str) -> list[str]:
    return re.findall(r"[a-z]+", name.lower())


def token_scores(name: str) -> list[float]:
    return [zipf_frequency(t, "en") for t in tokenize(name)]


def lexical_class(name: str) -> str:
    scores = token_scores(name)
    if not scores:
        return "empty"
    rare_ratio = sum(1 for s in scores if s < COMMON_THRESHOLD) / len(scores)
    if rare_ratio == 0.0:
        return "common"
    elif rare_ratio < 0.5:
        return "mixed"
    elif rare_ratio < 1.0:
        return "mostly_rare"
    else:
        return "non_standard"  # all tokens rare — jargon, proper nouns, invented terms


def min_zipf(name: str) -> float:
    scores = token_scores(name)
    return min(scores) if scores else 0.0


df["lexical_class"] = df["name"].apply(lexical_class)
df["min_zipf_freq"] = df["name"].apply(min_zipf)

print(df["lexical_class"].value_counts().to_string())
print(f"\nCoverage: {(df['lexical_class'] != 'non_standard').mean()*100:.1f}% of keywords have at least one common token")


lexical_class
common          47140
non_standard    21620
mostly_rare      8583
empty            5073
mixed            2426

Coverage: 74.5% of keywords have at least one common token


In [16]:
# Distribution of vocab classes
vc = df["lexical_class"].value_counts().reset_index()
vc.columns = ["lexical_class", "count"]

VC_COLORS = {
    "common":       "#2ca02c",
    "mixed":        "#1f77b4",
    "mostly_rare":  "#ff7f0e",
    "non_standard": "#d62728",
}

fig = px.pie(
    vc,
    names="lexical_class",
    values="count",
    color="lexical_class",
    color_discrete_map=VC_COLORS,
    title="Keyword vocabulary class breakdown",
)
fig.update_traces(textinfo="label+percent")
fig.show()


In [17]:
# Non-standard keywords — the 'mini language of cinema'
non_std = (
    df[df["lexical_class"] == "non_standard"]
    .sort_values("min_zipf_freq", ascending=True)
    [["tmdb_keyword_id", "name", "keyword_type", "min_zipf_freq"]]
    .head(60)
)
print(f"Most exotic keywords (lowest Zipf score):")
print(non_std.to_string(index=False))


Most exotic keywords (lowest Zipf score):
 tmdb_keyword_id                   name  keyword_type  min_zipf_freq
          294419               shardana theme_or_plot            0.0
          310929               fascisme theme_or_plot            0.0
          310986         perico cambeat theme_or_plot            0.0
          311053              barricada theme_or_plot            0.0
          311056 oberon cinematografica theme_or_plot            0.0
          311077                thiller theme_or_plot            0.0
          311103          unbrokenbonds theme_or_plot            0.0
          311197              shchedryk theme_or_plot            0.0
          311274                   cwrc theme_or_plot            0.0
          311290               varginha theme_or_plot            0.0
          311320                  bkbnl theme_or_plot            0.0
          311335                dongman theme_or_plot            0.0
          311373           epistemicide theme_or_plot        

In [18]:
# Cross: taxonomy category vs vocab class
cross = (
    df.groupby(["keyword_type", "lexical_class"])
    .size()
    .reset_index(name="count")
)

fig = px.bar(
    cross,
    x="keyword_type",
    y="count",
    color="lexical_class",
    color_discrete_map=VC_COLORS,
    barmode="stack",
    title="Vocabulary class by taxonomy category",
    labels={"keyword_type": "", "count": "Keywords", "lexical_class": "Vocab class"},
)
fig.update_layout(xaxis_tickangle=-30)
fig.show()


In [19]:
# Distribution of minimum Zipf scores across all keywords
fig = px.histogram(
    df,
    x="min_zipf_freq",
    nbins=50,
    color="lexical_class",
    color_discrete_map=VC_COLORS,
    title="Distribution of minimum token Zipf frequency per keyword",
    labels={"min_zipf_freq": "Min Zipf score (higher = more common)", "count": "Keywords"},
)
fig.add_vline(x=COMMON_THRESHOLD, line_dash="dash", line_color="black",
              annotation_text=f"threshold ({COMMON_THRESHOLD})", annotation_position="top right")
fig.update_layout(bargap=0.05)
fig.show()


## `is_narrative` — Story Content vs Utility Keywords

Filters on whether a keyword **describes the story world** of the film (plot, characters, themes, setting) vs production/format metadata that carries no emotional signal.

> From the TMDb guidelines: *"choose the few very best keywords to describe the plot of movie"*

| `keyword_type` | Signal | `is_narrative` |
|---|---|---|
| `theme_or_plot` | very high | ✅ True |
| `plot_event` | very high | ✅ True |
| `character_role` | high | ✅ True |
| `trope` | high | ✅ True |
| `object_motif` | medium | ✅ True |
| `mythic_figure` | medium | ✅ True |
| `location` | medium | ✅ True |
| `time_period` | low | ✅ True |
| `production_meta` | **none** | ❌ False |
| `genre_as_keyword` | **none** | ❌ False |

> Edge note: `serial killer` is not itself an emotion word, but it strongly predicts negative sentiment — narrative concepts leak emotional priors.

In [20]:
NON_NARRATIVE_CATEGORIES = {"production_meta", "genre_as_keyword"}

df["is_narrative"] = ~df["keyword_type"].isin(NON_NARRATIVE_CATEGORIES)

total = len(df)
narrative = df["is_narrative"].sum()
non_narrative = total - narrative

print(f"Narrative keywords     : {narrative:,} ({narrative/total*100:.1f}%)")
print(f"Non-narrative keywords : {non_narrative:,} ({non_narrative/total*100:.1f}%)")
print()
print("Non-narrative examples:")
print(df[~df["is_narrative"]][["tmdb_keyword_id", "name", "keyword_type"]].to_string(index=False))


Narrative keywords     : 84,693 (99.8%)
Non-narrative keywords : 149 (0.2%)

Non-narrative examples:
 tmdb_keyword_id                                                                                         name     keyword_type
             779                                                                                 martial arts genre_as_keyword
             818                                                                       based on novel or book  production_meta
            4326                                                                                 theater play  production_meta
            4344                                                                                      musical genre_as_keyword
            4434                                                                                    interview  production_meta
            5565                                                                                    biography  production_meta
          

In [21]:
# Narrative breakdown by category with is_narrative layered in
cat_narr = (
    df.groupby(["keyword_type", "is_narrative"])
    .size()
    .reset_index(name="count")
)
cat_narr["label"] = cat_narr["is_narrative"].map({True: "narrative", False: "non-narrative"})

fig = px.bar(
    cat_narr,
    x="count",
    y="keyword_type",
    color="label",
    orientation="h",
    color_discrete_map={"narrative": "#1f77b4", "non-narrative": "#d62728"},
    title="Narrative vs non-narrative by category",
    labels={"keyword_type": "", "count": "Keywords", "label": ""},
)
fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()


## Export — Enriched Keyword Dataset

Final enriched schema:

| Column | Type | Description |
|---|---|---|
| `tmdb_keyword_id` | int | TMDb canonical keyword ID |
| `name` | string | Keyword text |
| `keyword_type` | string | Taxonomy label |
| `lexical_class` | string | Vocabulary class (`common` → `non_standard`) |
| `min_zipf` | float | Minimum token Zipf frequency (rarity proxy) |
| `is_narrative` | bool | True = describes story world; False = production/format metadata |

In [22]:
import os
import shutil

EXPORT_DIR = Path("output/tmdb-keyword-enriched")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

enriched = df[[
    "tmdb_keyword_id", "name",
    "keyword_type", "lexical_class", "min_zipf_freq", "is_narrative",
]].copy()

out_path = EXPORT_DIR / "tmdb_keywords_enriched.csv"
enriched.to_csv(out_path, index=False)

# Include the notebook so consumers can see exactly how the data was produced
nb_src = Path("00_tmdb_keyword_download.ipynb")
shutil.copy(nb_src, EXPORT_DIR / nb_src.name)

print(f"Rows         : {len(enriched):,}")
print(f"Columns      : {list(enriched.columns)}")
print(f"CSV          : {out_path}  ({out_path.stat().st_size / 1024:.0f} KB)")
enriched.head(5)


Rows         : 84,842
Columns      : ['tmdb_keyword_id', 'name', 'keyword_type', 'lexical_class', 'min_zipf_freq', 'is_narrative']
CSV          : output/tmdb-keyword-enriched/tmdb_keywords_enriched.csv  (4351 KB)


,tmdb_keyword_id,name,keyword_type,lexical_class,min_zipf_freq,is_narrative
0,30,individual,theme_or_plot,common,5.00,True
1,65,holiday,theme_or_plot,common,4.66,True
2,74,germany,location,common,4.90,True
3,75,gunslinger,theme_or_plot,non_standard,2.58,True
4,83,saving the world,plot_event,common,4.59,True


In [23]:
# Upload to Kaggle via kagglehub (>= 0.4.1)
# Requires KAGGLE_TOKEN env var — generate at kaggle.com/settings.

import os

if not os.environ.get("KAGGLE_TOKEN"):
    print("Skipping upload: set KAGGLE_TOKEN env var to upload.")
else:
    import kagglehub

    kagglehub.login()  # reads KAGGLE_TOKEN from env
    user = kagglehub.whoami(verbose=False)["username"]
    handle = f"{user}/tmdb-keyword-enriched"

    run_date = export_date.strftime("%Y-%m-%d")
    kagglehub.dataset_upload(
        handle=handle,
        local_dataset_dir=str(EXPORT_DIR),
        version_notes=f"Daily export {run_date} — keyword_type, lexical_class, is_narrative",
        ignore_patterns=["*.json"],  # skip dataset-metadata.json
    )
    print(f"Uploaded to: kaggle.com/datasets/{handle}")


Skipping upload: set KAGGLE_TOKEN env var to upload.
